# ComfyUI + Pony Diffusion V6 XL + InstantID + IP-Adapter FaceID (Kaggle T4 x2)
Before running: **Settings -> Accelerator: GPU T4 x2**, **Internet: On**, and add a Kaggle secret named `CIVITAI_TOKEN` (Add-ons -> Secrets) and attach it to this notebook.
Run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install ComfyUI + custom nodes

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/comfyanonymous/ComfyUI
!git clone -q https://github.com/ltdrdata/ComfyUI-Manager ComfyUI/custom_nodes/ComfyUI-Manager
!git clone -q https://github.com/cubiq/ComfyUI_InstantID ComfyUI/custom_nodes/ComfyUI_InstantID
!git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!pip install -q -r ComfyUI/requirements.txt
!pip install -q insightface onnxruntime-gpu huggingface_hub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cf && chmod +x /kaggle/working/cf

## 2. Download models (about 17 GB)

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["CIVITAI_TOKEN"] = UserSecretsClient().get_secret("CIVITAI_TOKEN")
M = "/kaggle/working/ComfyUI/models"
for d in ["checkpoints","vae","controlnet","instantid","ipadapter","loras","clip_vision","insightface/models/antelopev2"]:
    os.makedirs(f"{M}/{d}", exist_ok=True)

def get(url, out):
    if os.path.exists(out) and os.path.getsize(out) > 1_000_000:
        print("have", out); return
    subprocess.run(["wget","-q","--show-progress","-O",out,url], check=True)

# Pony Diffusion V6 XL (Civitai model 257749, version 290640)
get(f"https://civitai.com/api/download/models/290640?type=Model&format=SafeTensor&size=pruned&fp=fp16&token={os.environ['CIVITAI_TOKEN']}",
    f"{M}/checkpoints/ponyDiffusionV6XL.safetensors")

HF = "https://huggingface.co"
# fp16-safe SDXL VAE
get(f"{HF}/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors", f"{M}/vae/sdxl_vae_fp16_fix.safetensors")
# InstantID
get(f"{HF}/InstantX/InstantID/resolve/main/ip-adapter.bin", f"{M}/instantid/ip-adapter.bin")
get(f"{HF}/InstantX/InstantID/resolve/main/ControlNetModel/diffusion_pytorch_model.safetensors", f"{M}/controlnet/instantid-controlnet.safetensors")
# IP-Adapter FaceID plus v2 (SDXL) + its LoRA
get(f"{HF}/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin", f"{M}/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin")
get(f"{HF}/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors", f"{M}/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors")
# CLIP vision (needed by FaceID plus v2)
get(f"{HF}/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/model.safetensors", f"{M}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")

# antelopev2 face models for InstantID (community mirror on Hugging Face)
from huggingface_hub import snapshot_download
snapshot_download("DIAMONIK7777/antelopev2", local_dir=f"{M}/insightface/models/antelopev2")
print("done")

In [ ]:
!find /kaggle/working/ComfyUI/models -type f -size +1M -exec ls -lh {} \; | awk '{print $5, $9}'
!ls /kaggle/working/ComfyUI/models/insightface/models/antelopev2

## 3. Launch ComfyUI (one instance per GPU) + public tunnel
A single image can't be split across two GPUs, so each GPU runs its own instance. Open both links to queue jobs in parallel.

In [ ]:
import subprocess, time, re, os
os.chdir("/kaggle/working/ComfyUI")
for i, port in [(0, 8188), (1, 8189)]:
    subprocess.Popen(f"python main.py --cuda-device {i} --port {port} --fp16-vae --listen 127.0.0.1 > /kaggle/working/comfy{i}.log 2>&1", shell=True)
    subprocess.Popen(f"/kaggle/working/cf tunnel --url http://127.0.0.1:{port} > /kaggle/working/cf{i}.log 2>&1", shell=True)
time.sleep(45)
for i in (0, 1):
    txt = open(f"/kaggle/working/cf{i}.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    print(f"GPU {i}:", m.group(0) if m else "tunnel not ready - rerun this cell")

## Usage tips
- **Pony prompts:** start with `score_9, score_8_up, score_7_up, ` and set CLIP skip to -2.
- **Settings:** 832x1216 or 1024x1024, DPM++ 2M Karras, 25-30 steps, CFG 6-7.
- **InstantID:** use the `InstantID Model Loader` (ip-adapter.bin), `InstantID Face Analysis` (provider CUDA), and the ControlNet `instantid-controlnet.safetensors`. Keep the InstantID weight around 0.8 and the ControlNet strength around 0.8 for Pony.
- **FaceID:** `IPAdapter Unified Loader FaceID` with the preset `FACEID PLUS V2`, plus the matching LoRA. buffalo_l downloads automatically on first use.
- Logs: `/kaggle/working/comfy0.log`, `comfy1.log`. If the tunnel shows nothing, rerun the last cell.
- Models are not kept between sessions, so rerun step 2 each time.